In [ ]:
# !pip install chromadb

In [ ]:
# LLM y Embeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

from chromadb import Chroma # Revisar

# Secretos
from dotenv import load_dotenv
import os

load_dotenv()

# Constantes 
API_KEY = os.getenv("GEMINI_API_KEY")
MODEL = 'gemini-2.5-flash'

In [ ]:
# LLM y Embeddings
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0, google_api_key=API_KEY)
embeddings = GoogleGenerativeAIEmbeddings(model=MODEL, google_api_key=API_KEY)

In [ ]:
# Base de conocimiento vectorial: ChromaDB
vectorstore = Chroma.from_documents(
    documents = documentos, # Generar documentos
    embedding = embeddings,
    collection_name = 'master_ia_doc' # Cambiar
)

In [ ]:
# Framework de agente: LangGraph (Ccon LangChain como base)
@tool
def buscar_documentacion(query: str) -> str:
    '''
    Busca información relevante en la documentación del **master de IA**.
    Úsala cuando necesites información sobre LangChain, RAG, LangGraph, embeddings, **AWS BedRock**
    '''
    docs = retriever.invoke(query)
    if not docs:
        return 'No enocntré información relevante sobre se tema'
    
    resultados = []

    for i, doc in enumerate(docs,1):
        fuente = doc.metadata.get('source', 'desconocida')
        resultados.append(f'[Fuente: {fuente}] \n {doc.page_content}')

    return '\n\n'.join(resultados)

In [ ]:
resultado = buscar_documentacion.invoke({'query': '¿Qué es el RAG'}) # Cambiar

In [ ]:
system_prompt = '''
Eres un asistente experto en **IA Generativa para un máster universitario**.
Tienes acceso a documentación del *máster**^. Cuando el usuario pregunta algo técnico, SIEMPRE usa la herramienta,
`buscar_documentacion` para fundamentar tu respuesta. Si la pregunta es de conversación general, puedes responder
directamente. 
Responde siempre en español
'''

In [ ]:
agente_rag = create_agent(
    model=llm,
    tools=[buscar_documentacion],
    system_prompt=system_prompt
)

In [ ]:
def preguntar_al_agente(pregunta: str):
    '''
    Función helper para hacer preguntas al agente y ver el proceso
    '''
    print('\n{"="*60}')
    print(f'PRegunta: {pregunta}')
    print('='*60)

    respuesta = agente_rag.invoke({
        'messages': [HumanMessage(content=pregunta)]
    })

    # Mostrar el proceso paso a paso
    print('\nProceso del agente:')
    for msg in respuesta['messages']:
        tipo = msg.__class__.__name__
        if tipo == 'HumanMessage':
            print(f'Usuario: {msg.content[:100]}')
        elif tipo == 'AIMessage':
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f'Agente decide usar herramientas: {tc['name']}')
                    print(f' Query: {tc['args'].get('query', '')}')

            else:
                print(f'\n Respuesta final: \n {msg.content}')
        elif tipo == 'ToolMessage':
            print(f'Resultado recuperado: {msg.content[:150]}...')